# Fully Supervised with Linear Embeddings

## Prerequisites

In [18]:
# install required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [19]:
# checking if the GPU is enabled
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

False


In [20]:
# using Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Representation Learning
## Making ResNet from scratch

SimCLR Training Procedure

Sample a batch of unlabeled images.

Apply two random augmentations to each image to create positive pairs.

Pass both views through the encoder and projection head.

Calculate contrastive loss for each positive pair.

Backpropagate and update the encoder and projection head weights.

After training, discard the projection head and use the encoder for downstream tasks

In [21]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

# forward pass
    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [33]:
class ResNet18(nn.Module):
    def __init__(self,zero_init_residual=False):
        super(ResNet18, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(512, 512)

        self.projection_head = nn.Sequential(nn.Linear(512, 512),
                                             nn.ReLU(inplace=True),
                                             nn.Linear(512, 128))

        for m in self.modules():
          if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
              nn.init.constant_(m.bias, 0)

        if zero_init_residual:
          for m in self.modules():
            if isinstance(m, BasicBlock):
              nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        z = self.projection_head(out)
        return out, z

        self.encoder = model
    def embedding(self, x):
        return self.encoder(x)


# return resnet18 model here !!!!
model = ResNet18().to(device)

# SimCLR
For each image in a batch, SimCLR generates two different augmented views (positive pair)

In [23]:
# generates two augmented views of the same image - for SimCLR contrastive learning
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x1 = self.transform(x)
        x2 = self.transform(x)
        return x1, x2

    def __len__(self):
        return len(self.dataset)

## Transformations + augmentations (for SimCLR)

In [24]:
"""
F.2 - the augmentations were:
- RandomHorizontalFlip
- RandomResizedCrop
- ColorJittering
- RandomGrayscaling
"""

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=32),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# for embeddings
embed_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# no test set needed
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
dataset = SimCLRDataset(trainset, transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4, drop_last=True)

classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck') # 10 classes

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Contrastive Loss Function (NT-Xent Loss)
Normalized Temperature-scaled Cross Entropy Loss.

For a batch of N images, there are 2N samples (each with two views).

Each positive pair is contrasted against 2(N - 1) negative pairs.

Loss encourages similar pairs to have high cosine similarity

In [25]:
# contrastive loss
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2N, 128)

    similarity = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    similarity /= temperature

    # extract positives from the FULL matrix before masking
    positives = torch.cat([
        torch.diag(similarity, N),
        torch.diag(similarity, -N)
    ], dim=0)  # (2N,)

    # mask self-similarity
    mask = torch.eye(2*N, dtype=torch.bool).to(z.device)
    similarity = similarity.masked_fill(mask, float('-inf'))

    logits = similarity  # (2N, 2N)
    labels = torch.cat([
        torch.arange(N, 2*N),  # first N samples match with last N
        torch.arange(0, N)     # last N samples match with first N
    ]).to(z.device)

    loss = F.cross_entropy(logits, labels)
    return loss

## Hyperparameters from SCAN
- SGD optimizer with 0.9 momentum
- Initial learning rate of 0.4 with a cosine scheduler
- Batch size was 512
- Weight decay of 0.0001

In [26]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4, nesterov=True)
#T_max should be num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=0)

In [10]:
# for loading a checkpoint only..!
"""
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint['loss']:.4f}")
"""

'\ncheckpoint_path = \'/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth\'\ncheckpoint = torch.load(checkpoint_path)\nmodel.load_state_dict(checkpoint[\'model_state_dict\'])\noptimizer.load_state_dict(checkpoint[\'optimizer_state_dict\'])\nscheduler.load_state_dict(checkpoint[\'scheduler_state_dict\'])\nstart_epoch = checkpoint[\'epoch\'] + 1\n\nprint(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint[\'loss\']:.4f}")\n'

## Training model

In [12]:
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
num_epochs = 500
best_loss = 1000 # just a high number

for epoch in range(num_epochs):
    model.train()
    running_loss = 0
    for (x1, x2) in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        # Forward pass
        _, z1 = model(x1)
        _, z2 = model(x2)
        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    avg_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # save the best model
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': avg_loss},
                   checkpoint_path)
        print(f"Saved best model - epoch {epoch+1} with loss: {avg_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


KeyboardInterrupt: 

In [27]:
# loading trained model once training has stopped
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu')

model = ResNet18().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
final_epoch = checkpoint['epoch'] + 1
print(f"Loaded final model (epoch {final_epoch}), loss: {checkpoint['loss']:.4f}")

Loaded final model (epoch 144), loss: 5.4876


In [28]:
model.eval()

ResNet18(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=

## Extracting embeddings
- forgot to implement embeddings method in ResNet18 training - next steps

In [29]:
# 50k imgs
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=embed_transform)
cifar_trainloader = torch.utils.data.DataLoader(cifar_train, batch_size=512, shuffle=False)
# 10k imgs
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=embed_transform)
cifar_testloader = torch.utils.data.DataLoader(cifar_test, batch_size=512, shuffle=False)

In [34]:
# train embeddings
train_embeddings = []
train_labels = []

with torch.no_grad():
    for imgs, labels in cifar_trainloader:
        imgs = imgs.to(device)
        embed = model.embedding(imgs)
        embed = F.normalize(embed, dim=1)
        train_embeddings.append(embed.cpu())
        train_labels.append(labels)
        print(f"Processed {len(train_embeddings) * 512} images")

train_embeddings_tensor = torch.cat(train_embeddings, dim=0)
train_labels_tensor = torch.cat(train_labels, dim=0)

AttributeError: 'ResNet18' object has no attribute 'encoder'

In [ ]:
# save train embeddings
torch.save({
    'embeddings': train_embeddings_tensor,  # 50k imgs and batch size
    'labels': train_labels_tensor
}, '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/train_embeddings.pth')

In [ ]:
# test embeddings
test_embeddings = []
test_labels = []

with torch.no_grad():
    for imgs, labels in cifar_testloader:
        imgs = imgs.to(device)
        embed = model.embedding(imgs)
        embed = F.normalize(embed, dim=1)
        test_embeddings.append(embed.cpu())
        test_labels.append(labels)
        print(f"Processed {len(test_embeddings) * 512} images")

test_embeddings_tensor = torch.cat(test_embeddings, dim=0)
test_labels_tensor = torch.cat(test_labels, dim=0)

In [ ]:
# save test embeddings
torch.save({
    'embeddings': test_embeddings_tensor,
    'labels': test_labels_tensor
}, '/content/drive/MyDrive/5CCSAMLF/F_Supervised/models/test_embeddings.pth')